In [5]:
import pandas as pd


In [6]:
covid = pd.read_csv('  https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv')
ebola = pd.read_csv('../data/raw/ebola_2014_2016_clean.csv')

In [7]:
print(covid.shape)
print(covid.columns)
covid.head()

(429435, 67)
Index(['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases',
       'new_cases_smoothed', 'total_deaths', 'new_deaths',
       'new_deaths_smoothed', 'total_cases_per_million',
       'new_cases_per_million', 'new_cases_smoothed_per_million',
       'total_deaths_per_million', 'new_deaths_per_million',
       'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients',
       'icu_patients_per_million', 'hosp_patients',
       'hosp_patients_per_million', 'weekly_icu_admissions',
       'weekly_icu_admissions_per_million', 'weekly_hosp_admissions',
       'weekly_hosp_admissions_per_million', 'total_tests', 'new_tests',
       'total_tests_per_thousand', 'new_tests_per_thousand',
       'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
       'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations',
       'people_vaccinated', 'people_fully_vaccinated', 'total_boosters',
       'new_vaccinations', 'new_vaccinations_smoothe

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN


In [8]:
print(ebola.shape)
print(ebola.columns)
ebola.head()

(2485, 4)
Index(['Country', 'Date',
       'Cumulative no. of confirmed, probable and suspected cases',
       'Cumulative no. of confirmed, probable and suspected deaths'],
      dtype='str')


,Country,Date,"Cumulative no. of confirmed, probable and suspected cases","Cumulative no. of confirmed, probable and suspected deaths"
0,Guinea,2014-08-29,648.0,430.0
1,Nigeria,2014-08-29,19.0,7.0
2,Sierra Leone,2014-08-29,1026.0,422.0
3,Liberia,2014-08-29,1378.0,694.0
4,Sierra Leone,2014-09-05,1261.0,491.0


In [9]:
covid.isnull().sum()

iso_code                                        0
continent                                   26525
location                                        0
date                                            0
total_cases                                 17631
                                            ...  
population                                      0
excess_mortality_cumulative_absolute       416024
excess_mortality_cumulative                416024
excess_mortality                           416024
excess_mortality_cumulative_per_million    416024
Length: 67, dtype: int64

In [10]:
ebola.isnull().sum()

Country                                                       0
Date                                                          0
Cumulative no. of confirmed, probable and suspected cases     8
Cumulative no. of confirmed, probable and suspected deaths    0
dtype: int64

In [11]:
ebola = ebola.rename(columns={
    'Country': 'country',
    'Date': 'date',
    'Cumulative no. of confirmed, probable and suspected cases': 'cases',
    'Cumulative no. of confirmed, probable and suspected deaths': 'deaths'
})

In [12]:
ebola['cases'] = ebola['cases'].ffill()

In [13]:
covid = covid.rename(columns={
    'location': 'country',
    'date': 'date',
    'total_cases': 'cases',
    'new_cases': 'new_cases',
    'total_deaths': 'deaths',
    'new_deaths': 'new_deaths',
    'population': 'population',
    'continent': 'continent'
})

In [14]:
covid = covid[['country', 'continent', 'date', 'cases', 'new_cases', 'deaths', 'new_deaths', 'population']]

In [15]:
covid['date'] = pd.to_datetime(covid['date'])
ebola['date'] = pd.to_datetime(ebola['date'])

In [16]:
covid['date'].dtype
ebola['date'].dtype

dtype('<M8[us]')

In [17]:
paises_covid = ['Brazil', 'Italy', 'South Korea']
covid_filtrado = covid[covid['country'].isin(paises_covid)]

paises_ebola = ['Guinea', 'Sierra Leone', 'Liberia']
ebola_filtrado = ebola[ebola['country'].isin(paises_ebola)]

In [18]:
ebola['country'].unique()

<ArrowStringArray>
[                  'Guinea',                  'Nigeria',
             'Sierra Leone',                  'Liberia',
                  'Senegal', 'United States of America',
                    'Spain',                     'Mali',
           'United Kingdom',                    'Italy']
Length: 10, dtype: str

In [ ]:
def calcular_dias_desde_inicio(df):
    df = df.copy()
    # data do primeiro caso registrado nesse país
    primeira_data = df.loc[df['cases'] > 0, 'date'].min()
    df['dias_desde_inicio'] = (df['date'] - primeira_data).dt.days
    return df

# aplicar separadamente para cada país (groupby + apply)
covid_filtrado = covid_filtrado.groupby('Italy', group_keys=False).apply(calcular_dias_desde_inicio)
covid_filtrado = covid_filtrado.groupby('Brazil', group_keys=False).apply(calcular_dias_desde_inicio)
covid_filtrado = covid_filtrado.groupby('South Korea', group_keys=False).apply(calcular_dias_desde_inicio)